# cisTarget database

In this notebook, we are creating a cisTarget motif database for the medulloblastoma mouse model single cell multiome data from:

[_Shiraishi, R. & Cancila, G. et al. (2024). Cancer-specific epigenome identifies oncogenic hijacking by nuclear factor I family proteins for medulloblastoma progression. Dev. Cell , 59:2302-2319._](https://www.cell.com/developmental-cell/fulltext/S1534-5807(24)00330-7)

This data set contains three samples comprising FACS sorted cells. These cells reflect the progression from healthy precursors to tumor cells:

1. Ptch1GNP: granule neuron precursors, P7
2. PNC: preneoplastic cells, P28
3. Tumor: tumor cells, adult mice

Data was downloaded from the Gene Expression Omnibus:

https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE240362

The `GSE240362_RAW.tar` file contains the raw gene expression counts files in HDF5 format (.h5) and ATAC fragment files. This is the typical output you would obtain from the [10X Genomics Cellranger software](https://www.10xgenomics.com/support/software/cell-ranger/latest/getting-started/cr-what-is-cell-ranger).

We are using the consensus peaks from the _pycisTopic_ processing as input:
* [pycistopic_atac_processing.ipynp](https://github.com/heckern/2026_ebi_workshop_scenicplus/pycistopic_atac_processing.ipynp)

For additional information on _pycisTarget_, see:
* https://pycistarget.readthedocs.io/en/latest/pycistarget_scenic%2B_human_brain.html#B.-Create-cisTarget-database

_cisTarget_ databases are used by SCENIC+ to link transcription factors to potential enhancer regions. In essence, this is a database that contains all predicted transcription factor motifs for a set of consensus regions and related scores.

## Creating conda environment

To create the _cisTarget_ database, it is easiest to set up a seperate conda environment.

```
conda create -n create_cistarget_databases     'python=3.10'     'numpy=1.22.4'     'pandas>=1.4.1'     'pyarrow>=7.0.0'     'numba>=0.55.1'     'python-flatbuffers'
conda activate create_cistarget_databases

```

## Downloading files

Before, we can create the database, we have to download the proper mouse genome (mm10 in our case) files.

```
wget https://hgdownload.cse.ucsc.edu/goldenpath/mm10/bigZips/mm10.fa.gz
wget https://hgdownload.cse.ucsc.edu/goldenpath/mm10/bigZips/mm10.chrom.sizes

gzip -d mm10.fa.gz

```

In addition, we should to download two tools:
* [_cbust_](https://pmc.ncbi.nlm.nih.gov/articles/PMC168947/): _cluster-bluster_ for scoring predicted TF binding site motifs
* [_bedtools_](https://bedtools.readthedocs.io/en/latest/): for working with genome coordinate files (BED format)

```
wget https://resources.aertslab.org/cistarget/programs/cbust
wget https://github.com/arq5x/bedtools2/releases/download/v2.30.0/bedtools.static.binary
```

_cbust_ needs a collection of TF binding site motifs as input which we download from the _cisTarget_ repository:

```
   wget https://resources.aertslab.org/cistarget/motif_collections/v10nr_clust_public/v10nr_clus
   unzip v10nr_clust_public.zip
```

## Preparing FASTA files

In the first step, we simply extracted genomic sequences for each _pycisTopic_ consensus region and are adding a default of 1kb padding sequence up- and downstream of the region.

```
set -e

if [ ! -e bedtools ]
then
	ln -s /storage/nhecker/software/bedtools.static.binary bedtools
fi
export PATH=$(pwd):$PATH


REGION_BED="/storage/nhecker/embl/GSE240362/pycistopic_mm10/consensus_peak_calling/consensus_regions.bed"
GENOME_FASTA="/storage/genomes/mm10/mm10.fa"
OUTPUT_FASTA="mm10.GSE240362.with_1kb_bg_padding.fa"
CHROMSIZES="/storage/genomes/mm10/mm10.chrom.sizes"
DATABASE_PREFIX="GSE240362_1kb_bg_with_mask"
SCRIPT_DIR="/storage/nhecker/git/create_cisTarget_databases"


${SCRIPT_DIR}/create_fasta_with_padded_bg_from_bed.sh \
        ${GENOME_FASTA} \
        ${CHROMSIZES} \
        ${REGION_BED} \
        ${OUTPUT_FASTA} \
        1000 \
        yes
```

## Creating the database

The actual motif database files are created with the `create_cistarget_motif_databases.py` command-line script using the cisTarget motif collection and the FASTA files as input. This step will usually take several hours to compute and many CPUs.

```
set -e

eval "$(conda shell.bash hook)"
conda activate create_cistarget_databases

CBUST_BIN_DIR=/storage/nhecker/software

export PATH=$PATH:$CBUST_BIN_DIR

OUT_DIR=""${PWD}""
CBDIR="/storage/nhecker/resources/cistarget_motif_db/v10nr_clust_public/singletons"
FASTA_FILE="/storage/nhecker/embl/GSE240362/cistargetdb_mm10/mm10.GSE240362.with_1kb_bg_padding.fa"
DATABASE_PREFIX="GSE240362_1kb_bg_with_mask"
MOTIF_LIST="motifs.txt"
SCRIPT_DIR="/storage/nhecker/git/create_cisTarget_databases"

ls ${CBDIR} > ${MOTIF_LIST}

"${SCRIPT_DIR}/create_cistarget_motif_databases.py" \
    -f ${FASTA_FILE} \
    -M ${CBDIR} \
    -m ${MOTIF_LIST} \
    -o ${OUT_DIR}/${DATABASE_PREFIX} \
    --bgpadding 1000 \
    -t 20
```

The output of the script are three database feather files:

In [4]:
! ls /storage/nhecker/embl/GSE240362/cistargetdb_mm10

GSE240362_1kb_bg_with_mask.motifs_vs_regions.scores.feather
GSE240362_1kb_bg_with_mask.regions_vs_motifs.rankings.feather
GSE240362_1kb_bg_with_mask.regions_vs_motifs.scores.feather
bash_create_fasta.sh
bash_create_motif_db.sh
bedtools
log_bash_create_fasta.log
log_bash_create_motif_db.log
mm10.GSE240362.with_1kb_bg_padding.fa
motifs.txt


* `GSE240362_1kb_bg_with_mask.motifs_vs_regions.scores.feather`: TF binding site motif score database file indexed by motifs (this file is not needed by SCENIC+)
* `GSE240362_1kb_bg_with_mask.regions_vs_motifs.scores.feather`: TF binding site motif score database file indexed by regions
* `GSE240362_1kb_bg_with_mask.regions_vs_motifs.rankings.feather`: TF binding site motif data base comprising rankings instead of scores 